# Lab 3: Contextual Bandit – User Classification

**Name:** Aryan Gosain  
**Roll Number:** U20230108  

This notebook so far covers:
- Data preprocessing
- User classification model (Context detection)

## 5.1 Data Pre-processing

- Load user and article datasets  
- Clean data (handle missing values)  
- Apply feature encoding for classification

In [8]:
# importing libraries
import pandas as pd
import numpy as np

# loading the datasets
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")
news_articles = pd.read_csv("data/news_articles.csv")

# checking the data shapes
print("train_users shape:", train_users.shape)
print("test_users shape:", test_users.shape)
print("news_articles shape:", news_articles.shape)

# checking for missing values
print("\nMissing values:")
print(train_users.isnull().sum())

train_users shape: (2000, 33)
test_users shape: (2000, 32)
news_articles shape: (209527, 6)

Missing values:
user_id                          0
age                            698
income                           0
clicks                           0
purchase_amount                  0
session_duration                 0
content_variety                  0
engagement_score                 0
num_transactions                 0
avg_monthly_spend                0
avg_cart_value                   0
browsing_depth                   0
revisit_rate                     0
scroll_activity                  0
time_on_site                     0
interaction_count                0
preferred_price_range            0
discount_usage_rate              0
wishlist_size                    0
product_views                    0
repeat_purchase_gap (days)       0
churn_risk_score                 0
loyalty_index                    0
screen_brightness                0
battery_percentage               0
cart_abandonment

In [ ]:
# removing duplicates
train_users = train_users.drop_duplicates()
test_users = test_users.drop_duplicates()
news_articles = news_articles.drop_duplicates()

# handling missing values 
# i saw that age has missing values, so filling with median
numeric_cols_list = train_users.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols_list:
    if col != "label":  
        median_val = train_users[col].median()
        train_users[col] = train_users[col].fillna(median_val)

for col in numeric_cols_list:
    if col in test_users.columns and col != "label":
        median_val = train_users[col].median()  # using train median
        test_users[col] = test_users[col].fillna(median_val)

# checking what columns we have
print("columns in train_users:", train_users.columns.tolist()[:5], "...")
print("\nmissing values after filling:", train_users.isnull().sum().sum())

columns in train_users: ['user_id', 'age', 'income', 'clicks', 'purchase_amount'] ...

 missing values after filling: 0


In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder

# getting feature columns
feature_cols = []
for col in train_users.columns:
    if col != "user_id" and col != "label":
        feature_cols.append(col)

print("total feature columns:", len(feature_cols))

# separating numeric and categorical columns
numeric_cols = []
categorical_cols = []

for col in feature_cols:
    if train_users[col].dtype in ['int64', 'float64']:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

print("numeric columns:", len(numeric_cols))
print("categorical columns:", categorical_cols)

# preparing X and y
X_full = train_users[feature_cols].copy()
y_full = train_users["label"].copy()

# encoding labels 
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_full)

print("\nlabel encoding:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label} -> {i}")

# scaling  features
scaler = StandardScaler()
X_numeric = X_full[numeric_cols].copy()
X_numeric_scaled = scaler.fit_transform(X_numeric)
X_numeric_scaled_df = pd.DataFrame(X_numeric_scaled, columns=numeric_cols)

# one hot encoding 
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=50)
X_categorical = X_full[categorical_cols].copy()
X_categorical_encoded = ohe.fit_transform(X_categorical)
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
X_categorical_df = pd.DataFrame(X_categorical_encoded, columns=cat_feature_names)

# combining everything
X_processed = pd.concat([X_numeric_scaled_df, X_categorical_df], axis=1)
print("\nfinal feature matrix shape:", X_processed.shape)

total feature columns: 31
numeric columns: 28
categorical columns: ['browser_version', 'region_code', 'subscriber']

label encoding:
user_1 -> 0
user_2 -> 1
user_3 -> 2

final feature matrix shape: (2000, 130)


## 5.2 User Classification (Context Detector)

- Split `train_users` into 80% training and 20% validation
- Train a classifier to predict user category (User1, User2, User3)
- Evaluate on validation set with `classification_report`

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# splitting data
# stratify to keep same proportion 
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("tryraining set size:", X_train_raw.shape[0])
print("validation set size:", X_val_raw.shape[0])

# scaling numeric features
X_train_numeric = scaler.fit_transform(X_train_raw[numeric_cols])
X_val_numeric = scaler.transform(X_val_raw[numeric_cols])
X_train_numeric_df = pd.DataFrame(X_train_numeric, columns=numeric_cols)
X_val_numeric_df = pd.DataFrame(X_val_numeric, columns=numeric_cols)

# encoding categorical features
X_train_cat = ohe.fit_transform(X_train_raw[categorical_cols])
X_val_cat = ohe.transform(X_val_raw[categorical_cols])
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
X_train_cat_df = pd.DataFrame(X_train_cat, columns=cat_feature_names)
X_val_cat_df = pd.DataFrame(X_val_cat, columns=cat_feature_names)

# combining
X_train = pd.concat([X_train_numeric_df, X_train_cat_df], axis=1)
X_val = pd.concat([X_val_numeric_df, X_val_cat_df], axis=1)

print("\nfinal training shape:", X_train.shape)
print("final validation shape:", X_val.shape)

tryraining set size: 1600
validation set size: 400

final training shape: (1600, 130)
final validation shape: (400, 130)


In [12]:
# training logistic regression classifier
print("training Logistic Regression classifier...")
classifier = LogisticRegression(max_iter=2000, random_state=42)
classifier.fit(X_train, y_train)

# prediction
y_pred = classifier.predict(X_val)
val_accuracy = accuracy_score(y_val, y_pred)
print(f"validation accuracy: {val_accuracy:.4f}")

training Logistic Regression classifier...
validation accuracy: 0.8900


In [ ]:
# classification report 
print("Classification Report on Validation Set:")
print(classification_report(
    y_val,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4
))

print("\nretraining on full training dataset for use on test_users...")

# preprocessing
X_full_numeric = scaler.fit_transform(X_full[numeric_cols])
X_full_cat = ohe.fit_transform(X_full[categorical_cols])
X_full_numeric_df = pd.DataFrame(X_full_numeric, columns=numeric_cols)
X_full_cat_df = pd.DataFrame(X_full_cat, columns=ohe.get_feature_names_out(categorical_cols))
X_full_processed = pd.concat([X_full_numeric_df, X_full_cat_df], axis=1)

# retraining classifier
classifier.fit(X_full_processed, y_encoded)
print(f"classifier retrained on all {X_full_processed.shape[0]} samples")

Classification Report on Validation Set:
              precision    recall  f1-score   support

      user_1     0.8378    0.8732    0.8552       142
      user_2     1.0000    0.8169    0.8992       142
      user_3     0.8529    1.0000    0.9206       116

    accuracy                         0.8900       400
   macro avg     0.8969    0.8967    0.8917       400
weighted avg     0.8998    0.8900    0.8898       400


 retraining on full training dataset for use on test_users...
classifier retrained on all 2000 samples


In [14]:
# function to predict user context for bandit system
# i will use this later 
def predict_user_context(user_df):
    test_numeric = scaler.transform(user_df[numeric_cols])
    test_cat = ohe.transform(user_df[categorical_cols])
    test_numeric_df = pd.DataFrame(test_numeric, columns=numeric_cols)
    test_cat_df = pd.DataFrame(test_cat, columns=ohe.get_feature_names_out(categorical_cols))
    test_processed = pd.concat([test_numeric_df, test_cat_df], axis=1)
    
    # predicting
    predictions = classifier.predict(test_processed)
    return predictions

# testing on test_users 
test_contexts = predict_user_context(test_users)
print("first 10 predictions:", test_contexts[:10])
print("\n distribution of predicted contexts:")
context_counts = pd.Series(test_contexts).value_counts().sort_index()
for i, count in context_counts.items():
    print(f"Context {i}: {count} users")

first 10 predictions: [2 0 0 0 0 2 2 0 2 2]

 distribution of predicted contexts:
Context 0: 609 users
Context 1: 666 users
Context 2: 725 users
